# Decoder Model Training with QLoRA (Qwen2.5-7B-Instruct)

This notebook fine-tunes a **decoder (causal LM)** model for Thai intent classification using **QLoRA** — 4-bit quantized LoRA.

### Encoder vs Decoder approach
| | Encoder (PhayaThaiBERT) | Decoder (Qwen2.5-7B) |
|---|---|---|
| Architecture | BERT — bidirectional encoder | GPT-style — autoregressive decoder |
| Output | Classification head → class logits | Generate text → `{"intent": "..."}` |
| Parameters | ~278M | ~7B |
| Memory trick | LoRA (FP16) | **QLoRA** (4-bit quantization + LoRA) |
| Trainer | `Trainer` | `SFTTrainer` (Supervised Fine-Tuning) |

### Why QLoRA?
- A 7B model in FP16 needs ~14 GB VRAM — doesn't fit on a 16 GB T4 with training overhead
- 4-bit quantization compresses the base model to ~4 GB
- LoRA adapters add ~0.5% trainable parameters on top
- Result: 7B model fine-tuning on a single T4 GPU

### Pipeline overview
1. Install dependencies
2. Imports
3. MLflow authentication
4. Configuration
5. Clean output directory
6. Load model with 4-bit quantization
7. Load tokenizer
8. Load and split dataset
9. Format data as chat messages
10. Configure LoRA
11. Configure training (SFTConfig)
12. Set up MLflow lineage callback
13. Create SFTTrainer
14. Train
15. Save the adapter

---
## 1. Install Dependencies

In [1]:
!pip install transformers==4.57.1 datasets peft==0.17.0 trl accelerate==1.10.0 bitsandbytes==0.48.1

In [2]:
!pip install "git+https://github.com/red-hat-data-services/mlflow@rhoai-3.3"

  Cloning https://github.com/red-hat-data-services/mlflow (to revision rhoai-3.3) to /tmp/pip-req-build-vue6_pe8
  Running command git clone --filter=blob:none --quiet https://github.com/red-hat-data-services/mlflow /tmp/pip-req-build-vue6_pe8
  Running command git checkout -b rhoai-3.3 --track origin/rhoai-3.3
  Switched to a new branch 'rhoai-3.3'
  branch 'rhoai-3.3' set up to track 'origin/rhoai-3.3'.
  Resolved https://github.com/red-hat-data-services/mlflow to commit c0f86528f988088bc07be8acc0db50074acee0f4
  Running command git submodule update --init --recursive -q
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


---
## 2. Imports

| Library | Purpose |
|---|---|
| `transformers` | `AutoModelForCausalLM`, `AutoTokenizer`, `BitsAndBytesConfig`, `TrainerCallback` |
| `trl` | `SFTTrainer` and `SFTConfig` — Supervised Fine-Tuning trainer for chat/instruction models |
| `peft` | `LoraConfig` — defines which layers get LoRA adapters |
| `bitsandbytes` | Backend for 4-bit quantization (QLoRA) |
| `datasets` | Load and split the JSONL/JSON dataset |
| `mlflow` | Experiment tracking — log params, metrics, dataset lineage |

In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainerCallback
import shutil
import os
import json
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset
import mlflow

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch: 2.7.1+cu128
CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB


---
## 3. MLflow Authentication

Set the MLflow tracking URI and authentication token before any MLflow operations.
- **`MLFLOW_TRACKING_TOKEN`** — your OpenShift token (from `oc whoami -t`)
- **`MLFLOW_TRACKING_URI`** — the MLflow server endpoint

In [4]:
from getpass import getpass

#os.environ["MLFLOW_TRACKING_TOKEN"] = getpass("Enter MLflow Token (from 'oc whoami -t'): ")
os.environ["MLFLOW_TRACKING_TOKEN"] = "sha256~nGtPiv_D_GRrkpwDYVPUU-BMEfYiLa3DiNvKcPdii0c"
os.environ["MLFLOW_TRACKING_URI"] = "https://mlflow.redhat-ods-applications.svc.cluster.local:8443"
os.environ["MLFLOW_WORKSPACE"] = "intent-classification-project"
os.environ["MLFLOW_RUN_NAME"] = "decoder-model-run"
os.environ["MLFLOW_TRACKING_HEADERS"] = "{\"X-MLFLOW-WORKSPACE\": \"intent-classification-project\", \"Content-Type\": \"application/json\"}"
os.environ["MLFLOW_EXPERIMENT_NAME"] = "decoder-model-finetuning"
os.environ["MLFLOW_TRACKING_INSECURE_TLS"] = "true"
print("Environment variables set!")

Environment variables set!


---
## 4. Configuration

| Parameter | Description |
|---|---|
| `MODEL_NAME` | Hugging Face model — a 7B instruction-tuned decoder |
| `DATA_PATH` | Path to the JSON training data |
| `OUTPUT_DIR` | Where checkpoints and the final adapter are saved |
| `DATASET_SOURCE` | Source identifier for MLflow data lineage |
| `RUN_NAME` | Name that appears in the MLflow experiment dashboard |

In [5]:
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
DATA_PATH = "data.json"
OUTPUT_DIR = "data/decoder-adapters"
DATASET_SOURCE = "data.json"
RUN_NAME = "decoder-model-run"
saved_model = "./model"

print(f"Model:      {MODEL_NAME}")
print(f"Data:       {DATA_PATH}")
print(f"Output:     {OUTPUT_DIR}")

Model:      Qwen/Qwen2.5-7B-Instruct
Data:       data.json
Output:     data/decoder-adapters


---
## 5. Clean Output Directory

Remove stale checkpoints from previous runs to avoid corruption.

In [6]:
if os.path.exists(OUTPUT_DIR):
    print(f"Cleaning existing output dir: {OUTPUT_DIR}")
    shutil.rmtree(OUTPUT_DIR)
else:
    print(f"Output dir does not exist yet, will be created during training.")

Cleaning existing output dir: data/decoder-adapters


---
## 6. Load Model with 4-bit Quantization (QLoRA)

### BitsAndBytesConfig

| Parameter | Value | Why |
|---|---|---|
| `load_in_4bit` | True | Compress 7B model from ~14 GB (FP16) to ~4 GB |
| `bnb_4bit_quant_type` | `"nf4"` | NormalFloat4 — optimized for normally distributed weights |
| `bnb_4bit_use_double_quant` | True | Quantize the quantization constants too — saves ~0.4 GB |
| `bnb_4bit_compute_dtype` | `float16` | Compute in FP16 during forward/backward (not 4-bit) |

After loading, we disable KV-cache (`use_cache = False`) because it's incompatible with gradient checkpointing used during training.

In [7]:
local_rank = int(os.environ.get("LOCAL_RANK", 0))
print(f"Local rank: {local_rank}")

Local rank: 0


In [8]:
import bitsandbytes as bnb

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

model_path = saved_model if os.path.isdir(saved_model) else MODEL_NAME
print(f"Loading model from: {model_path} (4-bit quantized)...")

if model_path == saved_model:
    _cfg_path = os.path.join(saved_model, "config.json")
    with open(_cfg_path) as f:
        _cfg = json.load(f)
    qc = _cfg.get("quantization_config", {})
    if qc.get("bnb_4bit_compute_dtype") != "float16":
        qc["bnb_4bit_compute_dtype"] = "float16"
        _cfg["quantization_config"] = qc
        with open(_cfg_path, "w") as f:
            json.dump(_cfg, f, indent=2)
        print("Patched saved config.json: bnb_4bit_compute_dtype → float16")

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    quantization_config=bnb_config,
    device_map={"": local_rank},
    dtype=torch.float16,
    trust_remote_code=True,
)

bf16_fixed = 0
for module in model.modules():
    if isinstance(module, bnb.nn.Linear4bit):
        if module.compute_dtype != torch.float16:
            module.compute_dtype = torch.float16
            bf16_fixed += 1
if bf16_fixed:
    print(f"Fixed {bf16_fixed} Linear4bit modules: compute_dtype → float16")

for name, param in model.named_parameters():
    if param.dtype == torch.bfloat16:
        param.data = param.data.to(torch.float16)
        print(f"  Cast {name} from bfloat16 → float16")

Loading model from: ./model (4-bit quantized)...


/opt/app-root/lib64/python3.12/site-packages/transformers/quantizers/auto.py:239: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


In [9]:
model.config.use_cache = False

if model_path != saved_model:
    model.save_pretrained(saved_model)
    print(f"Saved model to {saved_model} for future runs")

print(f"Model loaded on: {model.device}")
print(f"Model dtype: {model.dtype}")
print(f"Compute dtype: {bnb_config.bnb_4bit_compute_dtype}")

Model loaded on: cuda:0
Model dtype: torch.float16
Compute dtype: torch.float16


---
## 7. Load Tokenizer

The tokenizer must match the model. Two things we fix after loading:

1. **`pad_token = eos_token`** — many decoder models don't have a dedicated pad token; without this, batched training fails.

2. **Patch the chat template** — TRL's `assistant_only_loss=True` requires `{% generation %}` / `{% endgeneration %}` markers in the Jinja2 chat template to identify which tokens the model should learn to generate. Qwen's default template doesn't include these, so we inject them around assistant content.

In [10]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.save_pretrained(saved_model)

ORIGINAL_ASSISTANT_BLOCK = (
    "{{- '<|im_start|>' + message.role + '\\n' + message.content + '<|im_end|>' + '\\n' }}"
)
PATCHED_ASSISTANT_BLOCK = (
    "{{- '<|im_start|>' + message.role + '\\n' }}"
    "{% generation %}"
    "{{- message.content + '<|im_end|>' + '\\n' }}"
    "{% endgeneration %}"
)

if "{% generation %}" not in tokenizer.chat_template:
    tokenizer.chat_template = tokenizer.chat_template.replace(
        ORIGINAL_ASSISTANT_BLOCK, PATCHED_ASSISTANT_BLOCK
    )
    print("Patched chat template with {% generation %} markers for assistant_only_loss")
else:
    print("Chat template already has generation markers")

print(f"Vocab size:  {tokenizer.vocab_size}")
print(f"Pad token:   '{tokenizer.pad_token}' (id={tokenizer.pad_token_id})")
print(f"EOS token:   '{tokenizer.eos_token}' (id={tokenizer.eos_token_id})")

Patched chat template with {% generation %} markers for assistant_only_loss
Vocab size:  151643
Pad token:   '<|im_end|>' (id=151645)
EOS token:   '<|im_end|>' (id=151645)


---
## 8. Load and Split Dataset

Same 80/10/10 split as the encoder notebook. The test set is quarantined and saved separately.

In [11]:
dataset = load_dataset("json", data_files=DATA_PATH, split="train")

print(f"Total examples: {len(dataset)}")
print(f"Columns: {dataset.column_names}")

split_1 = dataset.train_test_split(test_size=0.2, seed=42)
split_2 = split_1["test"].train_test_split(test_size=0.5, seed=42)

train_dataset = split_1["train"]
val_dataset = split_2["train"]
test_dataset = split_2["test"]

print(f"\nTrain:      {len(train_dataset)} examples")
print(f"Validation: {len(val_dataset)} examples")
print(f"Test:       {len(test_dataset)} examples")

Total examples: 325
Columns: ['user_message', 'intent', 'session_history']

Train:      260 examples
Validation: 32 examples
Test:       33 examples


In [12]:
test_output_path = "data/decoder_test_split.jsonl"
os.makedirs("data", exist_ok=True)
test_dataset.to_json(test_output_path)
print(f"Test set saved to {test_output_path}")

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Test set saved to data/decoder_test_split.jsonl


---
## 9. Format Data as Chat Messages

Unlike the encoder approach (flat text → classification head), a decoder generates the answer as text. We format each example into the **chat message format** that instruction-tuned models expect:

```json
[
  {"role": "system", "content": "You are an AI intent classifier..."},
  {"role": "user", "content": "ช่วยเช็ค roaming data"},
  {"role": "assistant", "content": "ตอนนี้เหลือ 950MB"},
  {"role": "user", "content": "แล้วมันใช้ได้ถึงวันไหน"},
  {"role": "assistant", "content": "{\"intent\": \"MOBILE_ROAMING_CHECK_DATA_CURRENT\"}"}
]
```

The last assistant message is the **target output** — the model learns to generate this JSON.

### Model-specific handling
- **Qwen, Phi, DeepSeek** — support system messages natively
- **Gemma** — no system role; system prompt is prepended to the first user message

In [13]:
def format_messages_for_training(row: dict, base_model_id: str) -> dict:
    messages = []
    system_prompt = (
        "You are an AI intent classifier. Analyze the conversation history "
        "and the latest user message. You must output only a valid JSON object "
        "containing the predicted 'intent'."
    )

    model_name = base_model_id.lower()

    if "qwen" in model_name or "phi" in model_name or "deepseek" in model_name:
        messages.append({"role": "system", "content": system_prompt})
    elif "gemma" in model_name:
        pass
    else:
        messages.append({"role": "system", "content": system_prompt})

    for turn in row.get("session_history", []):
        hf_role = "assistant" if turn.get("role") == "assistant" else "user"
        messages.append({"role": hf_role, "content": turn.get("content", "")})

    current_msg = row.get("user_message", "")
    if "gemma" in model_name and len(messages) == 0:
        current_msg = f"{system_prompt}\n\n{current_msg}"

    messages.append({"role": "user", "content": current_msg})

    target_output = json.dumps({"intent": row.get("intent", "UNKNOWN")})
    messages.append({"role": "assistant", "content": target_output})

    return {"messages": messages}

### Preview the formatted messages

Let's see what the model will actually see during training.

In [14]:
sample_formatted = format_messages_for_training(train_dataset[1], MODEL_NAME)
for msg in sample_formatted["messages"]:
    print(f"[{msg['role']:>10}] {msg['content']}")

[    system] You are an AI intent classifier. Analyze the conversation history and the latest user message. You must output only a valid JSON object containing the predicted 'intent'.
[ assistant] ตรวจสอบแล้วค่ะ หมายเลข 0xx-xxx-xxxx ใช้แพ็กเกจ "เน็ตไม่อั้น 10Mbps" ราคา 599 บาทต่อเดือน แพ็กเกจจะรีเซ็ตวันที่ 15 เมษายน 2569 ค่ะ เหลืออีก 14 วันค่ะ
[      user] ใช้ไปเยอะแค่ไหนแล้วคะ
[ assistant] ใช้ไปแล้ว 43.8 GB ค่ะ แพ็กเกจนี้เป็นเน็ตไม่จำกัดค่ะ ไม่ต้องกังวลเน็ตหมดนะคะ
[      user] ดีเลยค่ะ ขอบคุณนะคะ
[ assistant] ยินดีค่ะ หากต้องการสอบถามเพิ่มเติม ติดต่อได้ตลอดนะคะ
[      user] เช็กแพ็กปัจจุบันให้หน่อยค่ะ เหลือกี่ GB กี่ ด.ถึงรีเซ็ต
[ assistant] {"intent": "MOBILE_PACKAGE_CHECK_DATA_CURRENT"}


### Apply formatting to train and validation sets

After formatting, each example has a single `messages` column (the chat format). The original columns (`user_message`, `intent`, `session_history`) are removed since `SFTTrainer` only needs the `messages` column.

In [15]:
original_columns = train_dataset.column_names

print("Formatting training dataset...")
train_dataset_formatted = train_dataset.map(
    lambda row: format_messages_for_training(row, MODEL_NAME),
    remove_columns=original_columns
)

print("Formatting validation dataset...")
val_dataset_formatted = val_dataset.map(
    lambda row: format_messages_for_training(row, MODEL_NAME),
    remove_columns=original_columns
)

print(f"\nTrain columns: {train_dataset_formatted.column_names}")
print(f"Val columns:   {val_dataset_formatted.column_names}")
print(f"\nSample messages count: {len(train_dataset_formatted[0]['messages'])}")

Formatting training dataset...
Formatting validation dataset...

Train columns: ['messages']
Val columns:   ['messages']

Sample messages count: 8


---
## 10. Configure LoRA

For decoder models, LoRA is applied to **all linear projections** in the transformer blocks — both the attention layers (Q, K, V, O) and the MLP layers (gate, up, down):

| Parameter | Value | Why |
|---|---|---|
| `r` | 16 | Rank of the low-rank matrices |
| `lora_alpha` | 32 (2×r) | Scaling factor |
| `lora_dropout` | 0.05 | Regularization |
| `bias` | "none" | Don't train bias terms |
| `task_type` | `CAUSAL_LM` | Causal language modeling (decoder) |
| `target_modules` | Q, K, V, O + gate, up, down | All linear projections — more capacity than encoder's Q+V only |

In [16]:
r = 16

peft_config = LoraConfig(
    r=r,
    lora_alpha=(2 * r),
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)

print(f"LoRA rank (r):      {peft_config.r}")
print(f"LoRA alpha:         {peft_config.lora_alpha}")
print(f"Target modules:     {peft_config.target_modules}")
print(f"Task type:          {peft_config.task_type}")

LoRA rank (r):      16
LoRA alpha:         32
Target modules:     {'down_proj', 'v_proj', 'o_proj', 'k_proj', 'q_proj', 'gate_proj', 'up_proj'}
Task type:          CAUSAL_LM


---
## 11. Training Arguments (SFTConfig)

`SFTConfig` extends `TrainingArguments` with SFT-specific options. Key settings:

### Batch size & memory
| Parameter | Value | Why |
|---|---|---|
| `per_device_train_batch_size` | 2 | Max that fits on T4 with 7B QLoRA + gradient checkpointing |
| `gradient_accumulation_steps` | 8 | Effective batch size = 2 × 8 = 16 |
| `gradient_checkpointing` | True | Trade compute for memory — recompute activations during backward |
| `fp16` | True | Half-precision training |

### SFT-specific
| Parameter | Value | Why |
|---|---|---|
| `max_length` | 2048 | Max token length per example (reduce to 1024 if OOM) |
| `packing` | False | Don't pack multiple examples into one sequence |
| `assistant_only_loss` | True | Only compute loss on assistant tokens — the model learns to generate answers, not memorize user messages |

### Learning rate & schedule
| Parameter | Value | Why |
|---|---|---|
| `learning_rate` | 2e-4 | Standard for LoRA fine-tuning |
| `warmup_ratio` | 0.1 | Ramp up LR for first 10% of steps |
| `lr_scheduler_type` | cosine | Smooth decay after warmup |
| `max_grad_norm` | 0.3 | Clip gradients |

### Evaluation & checkpointing
| Parameter | Value | Why |
|---|---|---|
| `eval_strategy` | steps | Evaluate at regular intervals |
| `eval_steps` | 0.1 | Every 10% of training |
| `load_best_model_at_end` | True | Revert to best checkpoint (lowest eval loss) |
| `save_total_limit` | 2 | Keep only 2 checkpoints to save disk |

In [17]:
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    max_length=1024,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=25,
    num_train_epochs=3,
    max_grad_norm=0.3,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    fp16=True,
    bf16=False,
    push_to_hub=False,
    packing=False,
    assistant_only_loss=True,

    report_to="mlflow",
    run_name=RUN_NAME,

    eval_strategy="steps",
    eval_steps=0.1,
    save_strategy="steps",
    save_steps=0.1,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    per_device_eval_batch_size=1,
    eval_accumulation_steps=1,
)

print(f"Batch size:         {training_args.per_device_train_batch_size}")
print(f"Grad accum steps:   {training_args.gradient_accumulation_steps}")
print(f"Effective batch:    {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"Learning rate:      {training_args.learning_rate}")
print(f"Epochs:             {training_args.num_train_epochs}")
print(f"Max length:         {training_args.max_length}")
print(f"Assistant-only loss: {training_args.assistant_only_loss}")

Batch size:         1
Grad accum steps:   4
Effective batch:    4
Learning rate:      0.0002
Epochs:             3
Max length:         1024
Assistant-only loss: True


---
## 12. MLflow Lineage Callback

Same purpose as the encoder notebook — inject dataset lineage and LoRA hyperparameters into the MLflow run that Hugging Face Trainer creates automatically.

In [18]:
class MLflowLineageCallback(TrainerCallback):
    def __init__(self, model_name, data_path, dataset_source, peft_config, train_ds, val_ds, test_ds):
        self.model_name = model_name
        self.data_path = data_path
        self.dataset_source = dataset_source
        self.peft_config = peft_config
        self.train_ds = train_ds
        self.val_ds = val_ds
        self.test_ds = test_ds

    def on_train_begin(self, args, state, control, **kwargs):
        if state.is_world_process_zero:
            print("Injecting data lineage into MLflow run...")

            mlflow.log_params({
                "model_name": self.model_name,
                "data_path": self.data_path,
                "lora_r": self.peft_config.r,
                "lora_alpha": self.peft_config.lora_alpha,
                "lora_dropout": self.peft_config.lora_dropout,
            })

            ds_train = mlflow.data.from_pandas(
                self.train_ds.to_pandas(),
                source=self.dataset_source, name="intent_train"
            )
            ds_val = mlflow.data.from_pandas(
                self.val_ds.to_pandas(),
                source=self.dataset_source, name="intent_val"
            )
            ds_test = mlflow.data.from_pandas(
                self.test_ds.to_pandas(),
                source=self.dataset_source, name="intent_test"
            )

            mlflow.log_input(ds_train, context="training")
            mlflow.log_input(ds_val, context="evaluation")
            mlflow.log_input(ds_test, context="test")

In [19]:
lineage_callback = MLflowLineageCallback(
    model_name=MODEL_NAME,
    data_path=DATA_PATH,
    dataset_source=DATASET_SOURCE,
    peft_config=peft_config,
    train_ds=train_dataset,
    val_ds=val_dataset,
    test_ds=test_dataset
)

print("MLflow lineage callback ready.")

MLflow lineage callback ready.


---
## 13. Create SFTTrainer

| Argument | What it provides |
|---|---|
| `model` | The 4-bit quantized base model |
| `args` | All hyperparameters from `SFTConfig` |
| `train_dataset` | Formatted training examples (chat messages) |
| `eval_dataset` | Formatted validation examples |
| `peft_config` | LoRA configuration — `SFTTrainer` applies it automatically |
| `processing_class` | The tokenizer |
| `callbacks` | MLflow lineage callback |

Note: Unlike the encoder's `Trainer`, `SFTTrainer` handles LoRA wrapping internally — you pass `peft_config` directly instead of calling `get_peft_model()` yourself.

In [20]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset_formatted,
    eval_dataset=val_dataset_formatted,
    peft_config=peft_config,
    processing_class=tokenizer,
    callbacks=[lineage_callback]
)

print("SFTTrainer ready.")

SFTTrainer ready.


In [21]:
print(f"Model dtype: {model.dtype}")
print(f"Training fp16: {training_args.fp16}, bf16: {training_args.bf16}")

Model dtype: torch.float16
Training fp16: True, bf16: False


In [ ]:
!accelerate env


Copy-and-paste the text below in your GitHub issue

- `Accelerate` version: 1.10.0
- Platform: Linux-5.14.0-570.96.1.el9_6.x86_64-x86_64-with-glibc2.34
- `accelerate` bash location: /opt/app-root/bin/accelerate
- Python version: 3.12.9
- Numpy version: 2.3.5
- PyTorch version: 2.7.1+cu128
- PyTorch accelerator: CUDA
- System RAM: 15.33 GB
- GPU type: Tesla T4
- `Accelerate` default config:
	Not found


In [ ]:
print("Starting training...")
trainer.train()

---
## 14. Train

Training loop for each of the 3 epochs:
1. Forward pass — generate tokens, compute cross-entropy loss (assistant tokens only)
2. Backward pass — gradients flow only through LoRA adapter matrices
3. Optimizer step — update adapter weights
4. Every 10% of steps — evaluate on validation set, save checkpoint if best eval loss

Metrics are logged to MLflow every 25 steps.

---
## 15. Save the LoRA Adapter

`trainer.save_model()` saves only the LoRA adapter weights — not the full 7B model. The adapter files are small (~10-50 MB).

To use the model later:
```python
from peft import PeftModel
base = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-7B-Instruct", ...)
model = PeftModel.from_pretrained(base, "data/decoder-adapters/latest")
```

In [ ]:
final_output_dir = f"{OUTPUT_DIR}/latest"

print(f"Saving adapter to {final_output_dir}...")
trainer.save_model(final_output_dir)

print(f"\nSaved files:")
for f in sorted(os.listdir(final_output_dir)):
    size = os.path.getsize(os.path.join(final_output_dir, f))
    if size > 1e6:
        print(f"  {f:<40} {size/1e6:.1f} MB")
    else:
        print(f"  {f:<40} {size/1e3:.1f} KB")

print("\nTraining complete!")

In [ ]:
!pip list | grep acc